In [2]:
import pandas as pd
import numpy as np
import json

In [3]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)
citation

{'doi': 'http://doi.org/10.1681/ASN.2018020125',
 'citation': 'Wu, H., Malone, A.F., Donnelly, E.L., Kirita, Y., Uchimura, K., Ramakrishnan, S.M., Gaut, J.P. and Humphreys, B.D., 2018. Single-cell transcriptomics of a human kidney allograft biopsy specimen defines a diverse inflammatory response. Journal of the American Society of Nephrology, 29(8), pp.2069-2080.',
 'tables': ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6065085/bin/ASN.2018020125SupplementaryData3.xls']}

## Define the metadata

In [4]:
organism = "homo_sapiens"
cell_source = "yolk_sac"
cell_state = None

reference = "GRCh38"

table_name = "clean_degs.xlsx" # local name

## Read in file

In [7]:
excel = pd.read_excel(table_name, sheet_name=None)
ct = {i: i.split('. ')[-1] for i in excel.keys()}

# stacks the sheets together and makes a new column "cell_type" from the sheet name
df = pd.concat(
    excel, keys=list(excel.keys())
    ).reset_index(0).rename(
        columns={"level_0": "celltype_id"}
        )
# # rename the cell types to be human readable
df["celltype"] = df["celltype_id"].map(ct)

In [8]:
df.head()

,celltype_id,gene,p_val,avg_logFC,pct.1,pct.2,p_val_adj,celltype
0,1. PT,GPX3,6.780000e-106,1.958635,0.442,0.109,1.390000e-101,PT
1,1. PT,CUBN,5.490000e-136,1.821094,0.417,0.046,1.120000e-131,PT
2,1. PT,CDH6,9.110000e-154,1.764531,0.464,0.035,1.870000e-149,PT
3,1. PT,LRP2,1.530000e-144,1.736607,0.480,0.049,3.140000e-140,PT
4,1. PT,PDZK1IP1,1.010000e-133,1.673290,0.431,0.038,2.060000e-129,PT


## Perform the filtering

In [12]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "pct.1"

In [13]:
min_mean = 100
max_pval = 1e-10
min_lfc = 1
max_gene_shares = 4
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

if col_rank:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)
else:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [14]:
markers[col_ct].value_counts()

Plasma2          20
LOH (DL)         20
PT               20
LOH (AL)         20
Pericyte         20
EC               20
B cells          20
Myofibroblast    20
CD               20
Mono2            20
Fibroblast       20
Mono1            20
Mast cells       20
Plasma1          15
T cells          14
Cycling cells    14
Name: celltype, dtype: int64

In [16]:
if col_rank:
    print(markers.groupby(col_ct)[col_rank].mean().sort_values())

celltype
T cells          0.327500
PT               0.440950
Cycling cells    0.457786
LOH (DL)         0.463800
EC               0.514700
Pericyte         0.527950
LOH (AL)         0.576600
Plasma1          0.604867
B cells          0.643500
CD               0.645600
Plasma2          0.647900
Mono2            0.657950
Myofibroblast    0.671700
Fibroblast       0.704150
Mono1            0.819900
Mast cells       0.918000
Name: pct.1, dtype: float64


## Convert to evidence json

In [17]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
save = df.to_dict(orient = "records")

In [18]:
save

[{'cell_type_label': 'T cells',
  'gene': 'CD96',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None},
 {'cell_type_label': 'Plasma2',
  'gene': 'IGLL1',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None},
 {'cell_type_label': 'T cells',
  'gene': 'TOX',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None},
 {'cell_type_label': 'Plasma1',
  'gene': 'IGKV4-1',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None},
 {'cell_type_label': 'T cells',
  'gene': 'BCL11B',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None},
 {'cell_type_label': 'Plasma2',
  'gene': 'IGLJ3',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None},
 {'cell_type_label': 'T cells',
  'gene': 'PPP1R16B',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None},
 {'cell_type_label': 'T cells',
  'gene': 'SYTL3',
  'organism': '

## Save evidence

In [19]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 